# Feature transforms and signal engineering

This notebook shows the `finstack_quant.features` API for common signal-engineering tasks: cross-sectional ranking, time-series transforms, grouped normalization, rolling pairwise metrics, and pandas DataFrame helpers.

The Rust-backed functions accept simple Python lists. The `finstack_quant.features.dataframe` helpers adapt the same transforms to pandas DataFrames and return Series or DataFrames aligned to the original index.

## Raw list transforms

Use the list API when you already have aligned arrays. `None` represents missing data; non-finite numeric inputs are treated as missing by the transform layer.

In [1]:
from finstack_quant.features import transform_cross_sectional, transform_timeseries

values = [1.0, 3.0, 2.0, None, 4.0, 5.0]
time_key = ["2026-01-02", "2026-01-02", "2026-01-02", "2026-01-03", "2026-01-03", "2026-01-03"]

print("rank:", transform_cross_sectional(values, time_key, "rank"))
print("zscore:", transform_cross_sectional(values, time_key, "zscore"))
print(
    "filled:",
    transform_cross_sectional(values, time_key, "fill_missing", {"value": 0.0}),
)

rank: [0.0, 1.0, 0.5, None, 0.0, 1.0]
zscore: [-1.224744871391589, 1.224744871391589, 0.0, None, -1.0, 1.0]
filled: [1.0, 3.0, 2.0, 0.0, 4.0, 5.0]


In [2]:
prices = [100.0, 101.5, 103.0, 99.0, 100.0, 104.0]
entity = ["A", "A", "A", "B", "B", "B"]
order = ["2026-01-02", "2026-01-03", "2026-01-04", "2026-01-02", "2026-01-03", "2026-01-04"]

print("returns:", transform_timeseries(prices, entity, order, "returns"))
print(
    "rolling mean:",
    transform_timeseries(prices, entity, order, "rolling_mean", {"window": 2}),
)
print("drawdown:", transform_timeseries(prices, entity, order, "drawdown"))

returns: [None, 0.014999999999999902, 0.014778325123152802, None, 0.010101010101010166, 0.040000000000000036]
rolling mean: [None, 100.75, 102.25, None, 99.5, 102.0]
drawdown: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## Build a small signal panel

The DataFrame helpers accept column names, index level names, or integer index level positions as key selectors. For a plain `DatetimeIndex`, cross-sectional `time_key` and time-series `order` can be omitted. For a `MultiIndex`, pass the relevant levels explicitly.

In [3]:
import pandas as pd

from finstack_quant.features import dataframe as fdf

dates = pd.bdate_range("2026-01-02", periods=6)
assets = ["ALPHA", "BETA", "GAMMA", "DELTA"]
sectors = {"ALPHA": "growth", "BETA": "growth", "GAMMA": "value", "DELTA": "value"}

rows = []
for date_idx, date in enumerate(dates):
    for asset_idx, asset in enumerate(assets):
        signal = 0.25 * date_idx + 0.4 * asset_idx
        price = 100.0 + 1.1 * date_idx + 2.0 * asset_idx + 0.3 * date_idx * asset_idx
        benchmark = 100.0 + 0.9 * date_idx
        beta = 0.7 + 0.1 * asset_idx
        volatility = 0.12 + 0.02 * asset_idx
        rows.append(
            {
                "date": date,
                "asset": asset,
                "sector": sectors[asset],
                "signal": signal,
                "price": price,
                "benchmark": benchmark,
                "beta": beta,
                "volatility": volatility,
            }
        )

df = pd.DataFrame(rows)
df.head(8)

,date,asset,sector,signal,price,benchmark,beta,volatility
0,2026-01-02,ALPHA,growth,0.00,100.0,100.0,0.7,0.12
1,2026-01-02,BETA,growth,0.40,102.0,100.0,0.8,0.14
2,2026-01-02,GAMMA,value,0.80,104.0,100.0,0.9,0.16
3,2026-01-02,DELTA,value,1.20,106.0,100.0,1.0,0.18
4,2026-01-05,ALPHA,growth,0.25,101.1,100.9,0.7,0.12
5,2026-01-05,BETA,growth,0.65,103.4,100.9,0.8,0.14
6,2026-01-05,GAMMA,value,1.05,105.7,100.9,0.9,0.16
7,2026-01-05,DELTA,value,1.45,108.0,100.9,1.0,0.18


## Cross-sectional and time-series features

Cross-sectional transforms partition by date. Time-series transforms group by asset and sort by date.

In [4]:
features = df[["date", "asset", "sector"]].copy()
features["signal_rank"] = fdf.cross_sectional(df, "signal", "date", "rank")
features["signal_zscore"] = fdf.cross_sectional(df, "signal", "date", "zscore")
features["sector_zscore"] = fdf.grouped(df, "signal", "date", "sector", "zscore")
features["price_return"] = fdf.timeseries(df, "price", "asset", "date", "returns")
features["rolling_price_z"] = fdf.timeseries(
    df,
    "price",
    "asset",
    "date",
    "rolling_zscore",
    {"window": 3, "min_periods": 2},
)

features.tail(8).round(4)

/var/folders/l1/s1m3_kfn43d77mc45c_3rv_h0000gn/T/ipykernel_94664/2283634488.py:15: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  features.tail(8).round(4)


,date,asset,sector,signal_rank,signal_zscore,sector_zscore,price_return,rolling_price_z
16,2026-01-08,ALPHA,growth,0.0000,-1.3416,-1.0,0.0106,1.0
17,2026-01-08,BETA,growth,0.3333,-0.4472,1.0,0.0132,1.0
18,2026-01-08,GAMMA,value,0.6667,0.4472,-1.0,0.0156,1.0
19,2026-01-08,DELTA,value,1.0000,1.3416,1.0,0.0179,1.0
20,2026-01-09,ALPHA,growth,0.0000,-1.3416,-1.0,0.0105,1.0
21,2026-01-09,BETA,growth,0.3333,-0.4472,1.0,0.0130,1.0
22,2026-01-09,GAMMA,value,0.6667,0.4472,-1.0,0.0153,1.0
23,2026-01-09,DELTA,value,1.0000,1.3416,1.0,0.0175,1.0


## Pairwise and portfolio-style signal helpers

`pairwise` computes rolling covariance, correlation, or beta between two aligned columns. Pipeline helpers such as `clean_signal`, `rank_to_weights`, `risk_scaled_weights`, and `neutralize_and_zscore` compose common cross-sectional signal-processing steps.

In [5]:
weights = df[["date", "asset", "signal", "beta", "volatility"]].copy()
weights["rolling_beta_to_benchmark"] = fdf.pairwise(
    df,
    "price",
    "benchmark",
    "asset",
    "date",
    "rolling_beta",
    {"window": 3, "min_periods": 2},
)
weights["clean_signal"] = fdf.clean_signal(df, "signal", "date", {"lower": 0.05, "upper": 0.95})
weights["rank_weight"] = fdf.rank_to_weights(df, "signal", "date")
weights["risk_scaled_weight"] = fdf.risk_scaled_weights(df, "signal", "date", "volatility")
weights["neutralized_zscore"] = fdf.neutralize_and_zscore(df, "signal", "date", ["beta"])

weights.tail(8).round(4)

/var/folders/l1/s1m3_kfn43d77mc45c_3rv_h0000gn/T/ipykernel_94664/2891990598.py:16: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  weights.tail(8).round(4)


,date,asset,signal,beta,volatility,rolling_beta_to_benchmark,clean_signal,rank_weight,risk_scaled_weight,neutralized_zscore
16,2026-01-08,ALPHA,1.00,0.7,0.12,1.2222,1.06,-0.375,0.1993,0.0
17,2026-01-08,BETA,1.40,0.8,0.14,1.5556,1.40,-0.125,0.2392,0.0
18,2026-01-08,GAMMA,1.80,0.9,0.16,1.8889,1.80,0.125,0.2691,0.0
19,2026-01-08,DELTA,2.20,1.0,0.18,2.2222,2.14,0.375,0.2924,0.0
20,2026-01-09,ALPHA,1.25,0.7,0.12,1.2222,1.31,-0.375,0.2142,0.0
21,2026-01-09,BETA,1.65,0.8,0.14,1.5556,1.65,-0.125,0.2424,0.0
22,2026-01-09,GAMMA,2.05,0.9,0.16,1.8889,2.05,0.125,0.2635,0.0
23,2026-01-09,DELTA,2.45,1.0,0.18,2.2222,2.39,0.375,0.2799,0.0


## Raw-list neutralization, normalization & panel pipeline

The list API also exposes `normalize_signal`, `neutralize` (cross-sectional OLS residualization against exposures), `transform_cross_sectional_grouped` (within-group transforms), `transform_timeseries_pairwise` (rolling beta/corr/cov between two columns), and `transform_panel` (a JSON pipeline of named operations).

In [6]:
import json

from finstack_quant.features import (
    neutralize,
    normalize_signal,
    transform_cross_sectional_grouped,
    transform_timeseries_pairwise,
    transform_panel,
)

sig = [1.0, 3.0, 2.0, 4.0, 5.0, 2.5]
tkey = ["d1", "d1", "d1", "d2", "d2", "d2"]
grp = ["X", "X", "Y", "X", "Y", "Y"]

print("normalize_signal (zscore):", [round(v, 3) for v in normalize_signal(sig, tkey, {"method": "zscore"})])
# Neutralize the signal against one exposure column (per-date OLS residuals)
expo = [[1.0, 0.5, 0.2, 1.0, 0.3, 0.7]]
print("neutralize (residuals):", [round(v, 3) for v in neutralize(sig, tkey, expo)])
print("cross_sectional_grouped (zscore within group):", transform_cross_sectional_grouped(sig, tkey, grp, "zscore"))

# Rolling pairwise beta of a signal vs a benchmark column, per entity
other = [100.0, 101.0, 102.0, 100.0, 99.0, 101.0]
ent = ["A", "A", "A", "B", "B", "B"]
order = ["1", "2", "3", "1", "2", "3"]
print("pairwise rolling_beta:", transform_timeseries_pairwise(sig, other, ent, order, "rolling_beta", {"window": 2, "min_periods": 1}))

# JSON panel pipeline: chain named operations and read back the result columns
panel_spec = json.dumps({
    "values": [100.0, 101.5, 103.0, 99.0, 100.0, 104.0],
    "entity": ent,
    "order": order,
    "operations": [
        {"name": "ret", "family": "timeseries", "op": "returns"},
        {"name": "roll2", "family": "timeseries", "op": "rolling_mean", "params": {"window": 2}},
    ],
})
print("transform_panel columns:", list(json.loads(transform_panel(panel_spec))["columns"]))

normalize_signal (zscore): [-1.225, 1.225, 0.0, 0.162, 1.136, -1.298]
neutralize (residuals): [-0.337, 0.898, -0.561, 0.73, 0.547, -1.277]
cross_sectional_grouped (zscore within group): [-1.0, 1.0, 0.0, 0.0, 1.0, -1.0]
pairwise rolling_beta: [None, 2.0, -1.0, None, -1.0, -1.25]
transform_panel columns: ['ret', 'roll2']


## Additional transforms (hampel, EWMA, rolling regression residual, robust)

A few more operations commonly used in production signal pipelines.


In [7]:
from finstack_quant.features import rolling_regression_residual

vals = [1.0, 3.0, 2.0, 10.0, 4.0, 5.0, 4.5, 4.2, 100.0, 4.8]
ent = ["A"]*10
ordr = [str(i) for i in range(10)]

print("hampel:", transform_timeseries(vals, ent, ordr, "hampel_filter", {"window": 5}))
print("ewma_mean:", transform_timeseries(vals, ent, ordr, "ewma_mean", {"span": 3}))
print("robust_zscore:", transform_cross_sectional(vals, ordr, "robust_zscore"))

expos = [[1.0]*10, [0.5]*10]
print("rolling_regression_residual:", rolling_regression_residual(vals, expos, ent, ordr, {"window": 5}))


hampel: [None, None, None, None, 4.0, 5.0, 4.5, 4.2, 4.5, 4.8]
ewma_mean: [1.0, 2.0, 2.0, 6.0, 5.0, 5.0, 4.75, 4.475, 52.2375, 28.518749999999997]
robust_zscore: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
rolling_regression_residual: [None, None, None, None, None, None, None, None, None, None]


## MultiIndex workflows

For MultiIndex panels, select key levels by name or position. This keeps the transform calls compact without copying key levels back into ordinary columns.

In [8]:
mi = df.set_index(["date", "asset", "sector"]).sort_index()

mi_features = pd.DataFrame(index=mi.index)
mi_features["rank_by_named_date"] = fdf.cross_sectional(mi, "signal", "date", "rank")
mi_features["rank_by_level_0"] = fdf.cross_sectional(mi, "signal", 0, "rank")
mi_features["asset_diff"] = fdf.timeseries(mi, "price", "asset", "date", "diff")
mi_features["sector_zscore"] = fdf.grouped(mi, "signal", "date", "sector", "zscore")

mi_features.tail(8).round(4)

rank_by_named_date  rank_by_level_0  asset_diff  \
date       asset sector                                                    
2026-01-08 ALPHA growth              0.0000           0.0000         1.1   
           BETA  growth              0.3333           0.3333         1.4   
           DELTA value               1.0000           1.0000         2.0   
           GAMMA value               0.6667           0.6667         1.7   
2026-01-09 ALPHA growth              0.0000           0.0000         1.1   
           BETA  growth              0.3333           0.3333         1.4   
           DELTA value               1.0000           1.0000         2.0   
           GAMMA value               0.6667           0.6667         1.7   

                         sector_zscore  
date       asset sector                 
2026-01-08 ALPHA growth           -1.0  
           BETA  growth            1.0  
           DELTA value             1.0  
           GAMMA value            -1.0  
2026-01-09 ALPHA growth           -1.0  
           BETA  growth            1.0  
           DELTA value             1.0  
           GAMMA value            -1.0

## Panel transform specifications

Use `panel` when several transforms share the same input column and should be returned as one DataFrame.

In [9]:
panel_features = fdf.panel(
    df,
    "signal",
    [
        {"name": "rank", "family": "cross_sectional", "op": "rank"},
        {"name": "zscore", "family": "cross_sectional", "op": "zscore"},
        {"name": "diff", "family": "timeseries", "op": "diff"},
    ],
    entity="asset",
    order="date",
    time_key="date",
)

panel_features.tail(8).round(4)

,rank,zscore,diff
16,0.0000,-1.3416,0.25
17,0.3333,-0.4472,0.25
18,0.6667,0.4472,0.25
19,1.0000,1.3416,0.25
20,0.0000,-1.3416,0.25
21,0.3333,-0.4472,0.25
22,0.6667,0.4472,0.25
23,1.0000,1.3416,0.25
